# MNIST LocalDpMod Parameter Search (Clipping, Sensitivity, Epsilon, Delta)

This notebook sweeps `LocalDpMod` settings for MNIST and ranks configurations by final `Global Test/accuracy`.

Note: in this repo, `LocalDpMod` is currently used in `strategy/dpfedaug/client_app.py`.


## Flower references

- https://flower.ai/docs/framework/ref-api/flwr.client.mod.LocalDpMod.html
- https://flower.ai/docs/framework/how-to-use-differential-privacy.html
- https://flower.ai/docs/framework/explanation-differential-privacy.html

Noise scale (docs/source):
`sigma = sensitivity * sqrt(2 * log(1.25 / delta)) / epsilon`


In [1]:
from __future__ import annotations

import itertools
import importlib.util
import os
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


In [2]:
def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "experiments/mnist/run_mnist_dpfedaug_experiments.py").exists():
            return p
    raise RuntimeError("Could not locate project root")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

RUNNER_FILE = PROJECT_ROOT / "experiments/mnist/run_mnist_dpfedaug_experiments.py"
spec = importlib.util.spec_from_file_location("mnist_dpfedaug_runner", RUNNER_FILE)
runner = importlib.util.module_from_spec(spec)
assert spec and spec.loader
spec.loader.exec_module(runner)

print("Project root:", PROJECT_ROOT)
print("Runner:", RUNNER_FILE)


Project root: <repo>
Runner: <repo>/experiments/mnist/run_mnist_dpfedaug_experiments.py


In [3]:
# Sweep controls
DRY_RUN = False
MAX_RUNS = None
SWEEP_ID = f"localdp_mnist_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
SWEEP_WANDB_PROJECT = f"DP-FedAug-MNIST-LocalDp-Sweep-{datetime.now().strftime('%Y%m%d_%H%M%S')}"

runner.WANDB_PROJECT = SWEEP_WANDB_PROJECT

FIXED = {
    "target_epsilon": None,
    "total_n": 600,
    "synthetic_count": 0,
    "partitioning": "dirichlet",
    "alpha": 1.0,
    "seed": 301,
    "balancing": "scaled",
}

GRID = {
    "updates_dp_enabled": [True],
    "updates_dp_epsilon": [0.5, 1.0, 2.0, 4.0],
    "updates_dp_delta": [1e-5, 1e-4],
    "updates_dp_clipping_norm": [0.25, 0.5, 1.0, 2.0],
    "updates_dp_sensitivity": [0.25, 0.5, 1.0, 2.0],
}

RESULTS_DIR = PROJECT_ROOT / "notebooks/nist/results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("SWEEP_ID:", SWEEP_ID)
print("W&B project:", runner.WANDB_PROJECT)


SWEEP_ID: localdp_mnist_20260211_091854
W&B project: DP-FedAug-MNIST-LocalDp-Sweep-20260211_091854


In [4]:
def build_grid_rows(grid: dict) -> list[dict]:
    keys = list(grid.keys())
    values = [grid[k] for k in keys]
    return [dict(zip(keys, combo)) for combo in itertools.product(*values)]

grid_rows = build_grid_rows(GRID)
if MAX_RUNS is not None:
    grid_rows = grid_rows[:MAX_RUNS]

plan = [{**FIXED, **row} for row in grid_rows]
plan_df = pd.DataFrame(plan)
print(f"Planned runs: {len(plan_df)}")
plan_df.head()


Planned runs: 128


,target_epsilon,total_n,synthetic_count,partitioning,alpha,seed,balancing,updates_dp_enabled,updates_dp_epsilon,updates_dp_delta,updates_dp_clipping_norm,updates_dp_sensitivity
0,None,600,0,dirichlet,1.0,301,scaled,True,0.5,0.00001,0.25,0.25
1,None,600,0,dirichlet,1.0,301,scaled,True,0.5,0.00001,0.25,0.50
2,None,600,0,dirichlet,1.0,301,scaled,True,0.5,0.00001,0.25,1.00
3,None,600,0,dirichlet,1.0,301,scaled,True,0.5,0.00001,0.25,2.00
4,None,600,0,dirichlet,1.0,301,scaled,True,0.5,0.00001,0.50,0.25


In [6]:
def run_one(exp: dict) -> dict:
    cfg_parts = runner.build_config_parts(
        target_epsilon=exp["target_epsilon"],
        total_n=exp["total_n"],
        synthetic_count=exp["synthetic_count"],
        partitioning=exp["partitioning"],
        seed=exp["seed"],
        balancing=exp["balancing"],
        updates_dp_enabled=exp["updates_dp_enabled"],
        updates_dp_epsilon=exp["updates_dp_epsilon"],
        updates_dp_delta=exp["updates_dp_delta"],
        updates_dp_clipping_norm=exp["updates_dp_clipping_norm"],
        updates_dp_sensitivity=exp["updates_dp_sensitivity"],
        alpha=exp.get("alpha"),
    )

    cmd = ["flwr", "run", ".", "--run-config", " ".join(cfg_parts)]
    env = runner.get_env_vars()

    t0 = time.time()
    proc = subprocess.run(cmd, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True, check=False)
    dt = time.time() - t0

    out = (proc.stdout or "") + "\n" + (proc.stderr or "")
    fail_markers = ["Traceback", "returned non-zero exit status", "ValueError:", "RuntimeError:"]
    has_failure_text = any(m in out for m in fail_markers)
    success = (proc.returncode == 0) and (not has_failure_text)

    return {
        **exp,
        "sweep_id": SWEEP_ID,
        "wandb_project": runner.WANDB_PROJECT,
        "returncode": proc.returncode,
        "success": success,
        "has_failure_text": has_failure_text,
        "duration_sec": round(dt, 2),
        "stdout_tail": "\n".join((proc.stdout or "").splitlines()[-25:]),
        "stderr_tail": "\n".join((proc.stderr or "").splitlines()[-25:]),
        "finished_utc": datetime.now(timezone.utc).isoformat(),
    }

run_records = []

if DRY_RUN:
    print("DRY_RUN=True, no experiments executed")
    display(plan_df.head())
else:
    original_pyproject = runner.modify_pyproject_for_dpfedaug()
    try:
        for i, exp in enumerate(plan, start=1):
            print(f"[{i}/{len(plan)}] clip={exp['updates_dp_clipping_norm']} sens={exp['updates_dp_sensitivity']} eps={exp['updates_dp_epsilon']} delta={exp['updates_dp_delta']}")
            rec = run_one(exp)
            run_records.append(rec)
            print(f"  rc={rec['returncode']} success={rec['success']} failure_text={rec['has_failure_text']} duration={rec['duration_sec']}s")
    finally:
        runner.restore_pyproject(original_pyproject)

runs_df = pd.DataFrame(run_records)
if not runs_df.empty:
    run_log_csv = RESULTS_DIR / f"{SWEEP_ID}_run_log.csv"
    runs_df.to_csv(run_log_csv, index=False)
    print("Saved run log:", run_log_csv)
    print(runs_df[["success", "has_failure_text", "returncode"]].value_counts(dropna=False).head())

runs_df.head()


[1/128] clip=0.25 sens=0.25 eps=0.5 delta=1e-05
  rc=0 success=True failure_text=False duration=304.68s
[2/128] clip=0.25 sens=0.5 eps=0.5 delta=1e-05
  rc=0 success=True failure_text=False duration=304.99s
[3/128] clip=0.25 sens=1.0 eps=0.5 delta=1e-05
  rc=0 success=True failure_text=False duration=310.4s
[4/128] clip=0.25 sens=2.0 eps=0.5 delta=1e-05
  rc=0 success=True failure_text=False duration=317.23s
[5/128] clip=0.5 sens=0.25 eps=0.5 delta=1e-05
  rc=0 success=True failure_text=False duration=316.65s
[6/128] clip=0.5 sens=0.5 eps=0.5 delta=1e-05
  rc=0 success=True failure_text=False duration=314.23s
[7/128] clip=0.5 sens=1.0 eps=0.5 delta=1e-05
  rc=0 success=True failure_text=False duration=315.25s
[8/128] clip=0.5 sens=2.0 eps=0.5 delta=1e-05
  rc=0 success=True failure_text=False duration=314.9s
[9/128] clip=1.0 sens=0.25 eps=0.5 delta=1e-05
  rc=0 success=True failure_text=False duration=315.92s
[10/128] clip=1.0 sens=0.5 eps=0.5 delta=1e-05
  rc=0 success=True failure_te

,target_epsilon,total_n,synthetic_count,partitioning,alpha,seed,balancing,updates_dp_enabled,updates_dp_epsilon,updates_dp_delta,...,updates_dp_sensitivity,sweep_id,wandb_project,returncode,success,has_failure_text,duration_sec,stdout_tail,stderr_tail,finished_utc
0,None,600,0,dirichlet,1.0,301,scaled,True,0.5,0.00001,...,0.25,localdp_mnist_20260211_091854,DP-FedAug-MNIST-LocalDp-Sweep-20260211_091854,0,True,False,304.68,Loading project configuration... \nSuccess\n\n...,[36m(ClientAppActor pid=283449)[0m 02/11/202...,2026-02-11T08:55:58.262391+00:00
1,None,600,0,dirichlet,1.0,301,scaled,True,0.5,0.00001,...,0.50,localdp_mnist_20260211_091854,DP-FedAug-MNIST-LocalDp-Sweep-20260211_091854,0,True,False,304.99,Loading project configuration... \nSuccess\n\n...,[36m(ClientAppActor pid=286950)[0m 02/11/202...,2026-02-11T09:01:03.256833+00:00
2,None,600,0,dirichlet,1.0,301,scaled,True,0.5,0.00001,...,1.00,localdp_mnist_20260211_091854,DP-FedAug-MNIST-LocalDp-Sweep-20260211_091854,0,True,False,310.40,Loading project configuration... \nSuccess\n\n...,[36m(ClientAppActor pid=290475)[0m 02/11/202...,2026-02-11T09:06:13.658317+00:00
3,None,600,0,dirichlet,1.0,301,scaled,True,0.5,0.00001,...,2.00,localdp_mnist_20260211_091854,DP-FedAug-MNIST-LocalDp-Sweep-20260211_091854,0,True,False,317.23,Loading project configuration... \nSuccess\n\n...,[36m(ClientAppActor pid=293979)[0m 02/11/202...,2026-02-11T09:11:30.890835+00:00
4,None,600,0,dirichlet,1.0,301,scaled,True,0.5,0.00001,...,0.25,localdp_mnist_20260211_091854,DP-FedAug-MNIST-LocalDp-Sweep-20260211_091854,0,True,False,316.65,Loading project configuration... \nSuccess\n\n...,[36m(ClientAppActor pid=297547)[0m 02/11/202...,2026-02-11T09:16:47.537936+00:00


In [ ]:
# Quick diagnostics
if not runs_df.empty:
    print("Failed rows:", int((~runs_df["success"]).sum()), "of", len(runs_df))
    if (~runs_df["success"]).any():
        display(runs_df.loc[~runs_df["success"], [
            "updates_dp_epsilon", "updates_dp_delta", "updates_dp_clipping_norm", "updates_dp_sensitivity", "stdout_tail", "stderr_tail"
        ]].head(3))


In [ ]:
# Pull from W&B project used for this sweep
# Requires online logging and WANDB_API_KEY

WANDB_ENTITY = None
WANDB_PROJECT = runner.WANDB_PROJECT

import wandb
api = wandb.Api(timeout=60)

if WANDB_ENTITY is None:
    WANDB_ENTITY = api.viewer.entity

path = f"{WANDB_ENTITY}/{WANDB_PROJECT}"
runs = api.runs(path=path)

rows = []
for r in runs:
    rows.append({
        "run_id": r.id,
        "state": r.state,
        "created_at": r.created_at,
        "accuracy": r.summary.get("Global Test/accuracy"),
        "loss": r.summary.get("Global Test/loss"),
        "auc": r.summary.get("Global Test/auc", r.summary.get("Global Test/AUC")),
        "updates_dp_epsilon": r.config.get("updates-dp-epsilon"),
        "updates_dp_delta": r.config.get("updates-dp-delta"),
        "updates_dp_clipping_norm": r.config.get("updates-dp-clipping-norm"),
        "updates_dp_sensitivity": r.config.get("updates-dp-sensitivity"),
        "total_n": r.config.get("total-n"),
        "partitioning": r.config.get("partitioning"),
        "non_iid_alpha": r.config.get("non-iid-alpha"),
        "seed": r.config.get("seed"),
    })

wb_df = pd.DataFrame(rows)
print(f"Fetched {len(wb_df)} W&B runs from {path}")

if not wb_df.empty:
    wb_df = wb_df.sort_values(["accuracy", "auc"], ascending=[False, False], na_position="last")
    wb_csv = RESULTS_DIR / f"{SWEEP_ID}_wandb_runs.csv"
    wb_df.to_csv(wb_csv, index=False)
    print("Saved W&B runs:", wb_csv)

wb_df.head(15)


In [ ]:
# Rank configurations
if 'wb_df' in globals() and not wb_df.empty:
    grp_cols = [
        "updates_dp_epsilon",
        "updates_dp_delta",
        "updates_dp_clipping_norm",
        "updates_dp_sensitivity",
    ]
    ranked = (
        wb_df.groupby(grp_cols, dropna=False)
        .agg(
            mean_accuracy=("accuracy", "mean"),
            std_accuracy=("accuracy", "std"),
            mean_auc=("auc", "mean"),
            n_runs=("run_id", "count"),
        )
        .reset_index()
        .sort_values(["mean_accuracy", "mean_auc"], ascending=False, na_position="last")
    )

    ranked_csv = RESULTS_DIR / f"{SWEEP_ID}_ranked_configs.csv"
    ranked.to_csv(ranked_csv, index=False)
    print("Saved ranked configs:", ranked_csv)
    display(ranked.head(20))
else:
    print("No W&B data available yet.")
    print("If runs were successful but W&B is empty, check WANDB_API_KEY/WANDB_MODE and sync state.")
